## **🎯 Objective** ##

- Create a Slowly Changing Dimension (Type 2) for sites that:

- Preserves historical site → province relationships

- Generates surrogate keys

- Supports time-travel analysis in Power BI


#### **Reason behind selecting Slowly Changing Dimension (Type 2):** ####
- The dim_site table is implementing SCD Type 2 because site attributes such as technology and vendor change over time and directly affect availability and SLA metrics. Preserving historical versions ensures analytical accuracy and allows meaningful before-and-after performance analysis.

In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import col, lit, expr, current_timestamp, max as spark_max, row_number
from pyspark.sql.window import Window


StatementMeta(, 2e9b4df1-1b25-4f11-ae4a-1e7f0edbcfe7, 3, Finished, Available, Finished)

In [2]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

print("="*70)
print("SCD TYPE 2 LOAD - DIM_SITE")
print("="*70)

# ============================================
# STEP 1: Get Last Load Date for Incremental Processing
# ============================================
print("\n📅 STEP 1: Determine Load Window")
print("-"*70)

try:
    last_load_result = spark.sql("""
        SELECT COALESCE(MAX(effective_start_date), DATE('1900-01-01')) as max_date
        FROM lh_Gold_Telecom.dbo.dim_site
    """).collect()[0]
    
    last_load_date = last_load_result['max_date']
    table_exists = True
    print(f"✓ dim_site exists")
    print(f"✓ Last load date: {last_load_date}")
    print(f"✓ Loading changes AFTER: {last_load_date}")
except:
    last_load_date = None
    table_exists = False
    print("✓ dim_site does not exist - performing INITIAL LOAD")

StatementMeta(, 2e9b4df1-1b25-4f11-ae4a-1e7f0edbcfe7, 4, Finished, Available, Finished)

SCD TYPE 2 LOAD - DIM_SITE

📅 STEP 1: Determine Load Window
----------------------------------------------------------------------
✓ dim_site exists
✓ Last load date: 2024-04-28
✓ Loading changes AFTER: 2024-04-28


In [3]:
# ============================================
# STEP 2: Extract Source Data (Incremental)
# ============================================
print("\n📥 STEP 2: Extract Source Data")
print("-"*70)

silver_df = spark.table("lh_Silver_Telecom.dbo.silver_network_events")

# Apply incremental filter if table exists
if table_exists and last_load_date:
    site_src = (
        silver_df
        .filter(col("outage_start_date") > last_load_date)
        .select(
            col("Site_ID").alias("site_id"),
            col("Province_Code"),
            col("Vendor"),
            col("Technology"),
            col("outage_start_date").alias("record_date")
        )
    )
    print(f"✓ Filtering records AFTER {last_load_date}")
else:
    site_src = (
        silver_df
        .select(
            col("Site_ID").alias("site_id"),
            col("Province_Code"),
            col("Vendor"),
            col("Technology"),
            col("outage_start_date").alias("record_date")
        )
    )
    print("✓ Loading ALL records (initial load)")

# Remove exact duplicates
site_src = site_src.dropDuplicates([
    "site_id",
    "Province_Code",
    "Vendor",
    "Technology",
    "record_date"
])

source_count = site_src.count()
print(f"✓ Extracted {source_count} source records")



StatementMeta(, 2e9b4df1-1b25-4f11-ae4a-1e7f0edbcfe7, 5, Finished, Available, Finished)


📥 STEP 2: Extract Source Data
----------------------------------------------------------------------
✓ Filtering records AFTER 2024-04-28
✓ Extracted 0 source records


In [4]:
# ============================================
# STEP 3: Join to Province Dimension
# ============================================
print("\n🔗 STEP 3: Enrich with Province Keys")
print("-"*70)

dim_province = spark.table("lh_Gold_Telecom.dbo.dim_province")

site_src = (
    site_src
    .join(
        dim_province,
        site_src.Province_Code == dim_province.province_code,
        "left"
    )
    .select(
        "site_id",
        "province_key",
        "Vendor",
        "Technology",
        "record_date"
    )
)

print("✓ Province dimension joined")

StatementMeta(, 2e9b4df1-1b25-4f11-ae4a-1e7f0edbcfe7, 6, Finished, Available, Finished)


🔗 STEP 3: Enrich with Province Keys
----------------------------------------------------------------------
✓ Province dimension joined


In [5]:
# ============================================
# STEP 4: Identify Changes (Latest per Site)
# ============================================
print("\n🔍 STEP 4: Identify New/Changed Sites")
print("-"*70)

# Keep only the LATEST change per site
w = Window.partitionBy("site_id").orderBy(col("record_date").desc())

site_latest = (
    site_src
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
    .select(
        col("site_id"),
        col("province_key"),
        col("Vendor").alias("vendor"),
        col("Technology").alias("technology"),
        col("record_date").alias("effective_start_date")
    )
)

print(f"✓ Latest state per site: {site_latest.count()} sites")

if table_exists:
    # Compare against current dimension state
    current_dim = spark.table("lh_Gold_Telecom.dbo.dim_site").filter(col("is_current") == True)
    
    changes = (
        site_latest.alias("src")
        .join(
            current_dim.alias("tgt"),
            on="site_id",
            how="left"
        )
        .filter(
            # New site (not in dimension)
            col("tgt.site_id").isNull() |
            # OR attribute changed
            (col("src.province_key") != col("tgt.province_key")) |
            (col("src.vendor") != col("tgt.vendor")) |
            (col("src.technology") != col("tgt.technology"))
        )
        .select(
            col("src.site_id"),
            col("src.province_key"),
            col("src.vendor"),
            col("src.technology"),
            col("src.effective_start_date")
        )
    )
    
    changes_count = changes.count()
    print(f"✓ Detected {changes_count} new/changed sites")
    
    if changes_count == 0:
        print("\n✅ No changes detected - exiting")
        
else:
    # Initial load - all sites are "changes"
    changes = site_latest
    print(f"✓ Initial load: {changes.count()} sites to insert")

StatementMeta(, 2e9b4df1-1b25-4f11-ae4a-1e7f0edbcfe7, 7, Finished, Available, Finished)


🔍 STEP 4: Identify New/Changed Sites
----------------------------------------------------------------------
✓ Latest state per site: 0 sites
✓ Detected 0 new/changed sites

✅ No changes detected - exiting


In [6]:
# ============================================
# STEP 5: Create Temporary View for MERGE
# ============================================
print("\n📝 STEP 5: Prepare Changes for MERGE")
print("-"*70)

changes.createOrReplaceTempView("changes_temp")
print("✓ Temporary view created: changes_temp")


StatementMeta(, 2e9b4df1-1b25-4f11-ae4a-1e7f0edbcfe7, 8, Finished, Available, Finished)


📝 STEP 5: Prepare Changes for MERGE
----------------------------------------------------------------------
✓ Temporary view created: changes_temp


In [7]:
# ============================================
# Execute SCD Type 2 MERGE
# ============================================

from pyspark.sql.functions import lit

print("\n🔄 STEP 6: Execute SCD Type 2 MERGE")
print("-"*70)

if table_exists:
    # ============================================
    # INCREMENTAL LOAD - MERGE EXISTING TABLE
    # ============================================
    
    # Execute MERGE to close old versions and insert new sites
    merge_result = spark.sql("""
        MERGE INTO lh_Gold_Telecom.dbo.dim_site AS tgt
        USING changes_temp AS src
        ON tgt.site_id = src.site_id 
          AND tgt.is_current = true
        
        -- When site exists AND attributes changed
        WHEN MATCHED AND (
          tgt.province_key <> src.province_key OR
          tgt.vendor <> src.vendor OR
          tgt.technology <> src.technology
        ) THEN UPDATE SET
          is_current = false,
          effective_end_date = date_sub(src.effective_start_date, 1)
        
        -- When site doesn't exist (new site)
        WHEN NOT MATCHED THEN INSERT (
          site_id,
          province_key,
          vendor,
          technology,
          effective_start_date,
          effective_end_date,
          is_current
        ) VALUES (
          src.site_id,
          src.province_key,
          src.vendor,
          src.technology,
          src.effective_start_date,
          DATE('9999-12-31'),
          true
        )
    """)
    
    print("✓ MERGE completed")
    
    # ============================================
    # Insert New Versions for Changed Sites
    # ============================================
    print("\n➕ STEP 7: Insert New Versions")
    print("-"*70)
    
    # Find sites that were updated (is_current was set to false)
    updated_sites = spark.sql("""
        SELECT DISTINCT src.site_id
        FROM changes_temp src
        INNER JOIN lh_Gold_Telecom.dbo.dim_site tgt
          ON src.site_id = tgt.site_id
        WHERE tgt.is_current = false
          AND tgt.effective_end_date < DATE('9999-12-31')
    """)
    
    # Insert new versions for these sites
    new_versions = (
        changes
        .join(updated_sites, on="site_id", how="inner")
        .withColumn("effective_end_date", lit("9999-12-31").cast("date"))
        .withColumn("is_current", lit(True))
    )
    
    insert_count = new_versions.count()
    
    if insert_count > 0:
        (
            new_versions
            .write
            .format("delta")
            .mode("append")
            .saveAsTable("lh_Gold_Telecom.dbo.dim_site")
        )
        print(f"✓ Inserted {insert_count} new versions")
    else:
        print("✓ No new versions to insert (only new sites added)")

else:
    # ============================================
    # INITIAL LOAD - TABLE CREATION
    # ============================================
    # This block only runs when dim_site doesn't exist
    # The table is CREATED by the saveAsTable() operation
    # ============================================
    print("✓ Performing initial load...")
    
    initial_load = (
        changes
        .withColumn("effective_end_date", lit("9999-12-31").cast("date"))
        .withColumn("is_current", lit(True))
    )
    
    # TABLE IS CREATED HERE with inferred schema
    (
        initial_load
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("lh_Gold_Telecom.dbo.dim_site")
    )
    
    print(f"✓ Initial load complete: {initial_load.count()} sites inserted")
    print(f"✓ Table created: lh_Gold_Telecom.dbo.dim_site")

print("\n" + "="*70)
print("✅ COMPLETED")
print("="*70)

StatementMeta(, 2e9b4df1-1b25-4f11-ae4a-1e7f0edbcfe7, 9, Finished, Available, Finished)


🔄 STEP 6: Execute SCD Type 2 MERGE
----------------------------------------------------------------------
✓ MERGE completed

➕ STEP 7: Insert New Versions
----------------------------------------------------------------------
✓ No new versions to insert (only new sites added)

✅ COMPLETED



## Expire Old Records (Close Previous Versions)

### What this does:
 - For sites that changed, closes out the old version
 - Sets end_date to day before new version starts
 - Sets is_current flag to False
 
### SCD Type 2 Logic:
 - Keep old record with historical data intact
 - Mark it as expired by setting end_date and is_current=False
 - This maintains complete audit trail of changes


In [8]:
# ============================================
# Data Quality Validation
# ============================================
print("\n✅ Data Quality Validation")
print("-"*70)

dim_site_final = spark.table("lh_Gold_Telecom.dbo.dim_site")

# Check 1: One current record per site
duplicate_current = (
    dim_site_final
    .filter(col("is_current") == True)
    .groupBy("site_id")
    .count()
    .filter(col("count") > 1)
)

dup_count = duplicate_current.count()

if dup_count > 0:
    print(f"❌ ERROR: {dup_count} sites have multiple current records!")
    duplicate_current.show(20, truncate=False)
    raise Exception("SCD VIOLATION: Multiple current rows per site")
else:
    print("✅ PASS: Exactly one current record per site")

# Check 2: Current records end date validation
invalid_end_dates = (
    dim_site_final
    .filter(col("is_current") == True)
    .filter(col("effective_end_date") != lit("9999-12-31").cast("date"))
)

if invalid_end_dates.count() > 0:
    print("❌ ERROR: Current records have invalid end dates")
    invalid_end_dates.show(20, truncate=False)
    raise Exception("SCD VIOLATION: Invalid end dates on current records")
else:
    print("✅ PASS: All current records end at 9999-12-31")

# Check 3: Date logic validation
invalid_dates = (
    dim_site_final
    .filter(
        (col("effective_end_date") != lit("9999-12-31").cast("date")) &
        (col("effective_end_date") < col("effective_start_date"))
    )
)

if invalid_dates.count() > 0:
    print("❌ ERROR: Invalid date ranges detected")
    invalid_dates.show(20, truncate=False)
    raise Exception("SCD VIOLATION: End date before start date")
else:
    print("✅ PASS: All date ranges are valid")

StatementMeta(, 2e9b4df1-1b25-4f11-ae4a-1e7f0edbcfe7, 10, Finished, Available, Finished)


✅ Data Quality Validation
----------------------------------------------------------------------
✅ PASS: Exactly one current record per site
✅ PASS: All current records end at 9999-12-31
✅ PASS: All date ranges are valid


In [9]:
# ============================================
# Summary Statistics
# ============================================
print("\n" + "="*70)
print("📊 FINAL SUMMARY")
print("="*70)

stats = dim_site_final.select(
    count("*").alias("total_records"),
    countDistinct("site_id").alias("unique_sites"),
    sum(when(col("is_current"), 1).otherwise(0)).alias("current_records"),
    sum(when(~col("is_current"), 1).otherwise(0)).alias("historical_records")
).collect()[0]

print(f"\nTotal Records:      {stats['total_records']:,}")
print(f"Unique Sites:       {stats['unique_sites']:,}")
print(f"Current Records:    {stats['current_records']:,}")
print(f"Historical Records: {stats['historical_records']:,}")

sites_with_history = (
    dim_site_final
    .groupBy("site_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(f"Sites with History: {sites_with_history:,}")

print("\n" + "="*70)
print("🎉 SCD TYPE 2 LOAD COMPLETED SUCCESSFULLY")
print("="*70)

StatementMeta(, 2e9b4df1-1b25-4f11-ae4a-1e7f0edbcfe7, 11, Finished, Available, Finished)


📊 FINAL SUMMARY

Total Records:      1,000
Unique Sites:       1,000
Current Records:    1,000
Historical Records: 0
Sites with History: 0

🎉 SCD TYPE 2 LOAD COMPLETED SUCCESSFULLY
